In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import BatchEncoding
from transformers import AutoTokenizer, AutoModel

In [3]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
split_dir = os.path.join(preprocessed_dir, "split_features")
text_path = os.path.join(split_dir, "df_text.csv")

# Load cleaned text data
df_text = pd.read_csv(text_path)
print(f"Loaded df_text.csv with shape: {df_text.shape}")
df_text.head()

Loaded df_text.csv with shape: (3730329, 3)


,item_id,item_title,item_entity_names
0,287924,"消费金-百岁山天然矿泉水整箱小瓶饮用水,绿账优淘,天然矿泉水整箱小瓶付邮购","饮用水,百岁山,天然矿泉,绿账,消费金,小瓶,付邮购"
1,425272,百万医疗免费体验,"医疗,好医保,百万医疗"
2,604771,逛15秒商品橱窗,商品橱窗
3,237488,中国联通营业厅电话卡,"阿里宝卡,宝卡,流量卡,vip卡"
4,604771,逛15秒商品橱窗,商品橱窗


In [4]:
# Set device -> Switched to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Load Pre-trained BERT Tokenizer & Model

HuggingFace `transformers` library is used to load the pre-trained `bert-base-chinese` model and tokeniser. This model is well-suited for handling Chinese-language input, including product titles and semantic tags.

The model is loaded in `eval()` mode to disable gradient tracking during inference.

In [5]:
# Load pre-trained Chinese BERT
bert_model_name = "bert-base-chinese"

tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
model = AutoModel.from_pretrained(bert_model_name)
model.to(device)
model.eval()  # Disable training mode

print("BERT model and tokenizer loaded:", bert_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

BERT model and tokenizer loaded: bert-base-chinese


Although the text data was preprocessed earlier, we reapply basic cleaning steps here to ensure consistency. This is especially useful if the notebook is run independently.

In [12]:
# Text columns to clean
text_columns = ['item_title', 'item_entity_names']

for col in text_columns:
    df_text[col] = df_text[col].fillna("unknown")
    df_text[col] = df_text[col].astype(str).str.strip()
    df_text[col] = df_text[col].str.lower()  # BERT is uncased for Chinese

print("Re-cleaned text fields:")
df_text[text_columns].head()

Re-cleaned text fields:


,item_title,item_entity_names
0,"消费金-百岁山天然矿泉水整箱小瓶饮用水,绿账优淘,天然矿泉水整箱小瓶付邮购","饮用水,百岁山,天然矿泉,绿账,消费金,小瓶,付邮购"
1,百万医疗免费体验,"医疗,好医保,百万医疗"
2,逛15秒商品橱窗,商品橱窗
3,中国联通营业厅电话卡,"阿里宝卡,宝卡,流量卡,vip卡"
4,逛15秒商品橱窗,商品橱窗


## Tokenization & Embedding

We generate semantic embeddings for two text fields:

- `item_title`: Free-text product title
- `item_entity_names`: Structured entity tags

The process:
1. Tokenize using HuggingFace tokenizer with padding/truncation
2. Feed through pre-trained BERT (in eval mode)
3. Extract the `[CLS]` token as the sentence-level embedding (shape: `[batch_size, 768]`)
4. Store results as NumPy arrays for later use

In [14]:
# Custom Dataset to wrap list of texts for use with PyTorch DataLoader
class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts  # List of input strings

    def __len__(self):
        return len(self.texts)  # Total number of items

    def __getitem__(self, idx):
        return self.texts[idx]  # Return single item at index `idx`

In [20]:
# Function to batch-tokenize and extract [CLS] embeddings using BERT
def get_bert_embeddings(text_list, tokenizer, model, batch_size=64, max_length=32):
    # Wrap list of strings as a PyTorch dataset
    dataset = TextDataset(text_list)

    # Create DataLoader to batch inputs (faster, more memory efficient)
    dataloader = DataLoader(dataset, batch_size=batch_size)

    all_cls_embeddings = []  # Will store [CLS] embeddings from each batch

    # Disable gradient tracking for inference
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Encoding with BERT"):
            # Tokenize the batch of text
            encoding = tokenizer(
                batch,
                padding=True,              # Pad shorter sequences
                truncation=True,           # Truncate longer ones
                max_length=max_length,     # Set max token length
                return_tensors='pt'        # Return PyTorch tensors
            ).to(device)                   # Move to GPU (or CPU)

            # Forward pass through BERT model
            outputs = model(**encoding)

            # Extract only the [CLS] token embeddings: shape [batch_size, hidden_size]
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] is first token

            # Move embeddings to CPU and store
            all_cls_embeddings.append(cls_embeddings.cpu())

    # Concatenate all batches into a single NumPy array
    return torch.cat(all_cls_embeddings, dim=0).numpy()

In [ ]:
# Generate BERT embeddings
title_embeddings = get_bert_embeddings(df_text['item_title'].tolist(), tokenizer, model)

print("item_title embedding shape:", title_embeddings.shape)


Encoding with BERT: 100%|██████████| 58287/58287 [27:55<00:00, 34.79it/s]


In [21]:
entity_embeddings = get_bert_embeddings(df_text['item_entity_names'].tolist(), tokenizer, model)

print("item_entity_names embedding shape:", entity_embeddings.shape)

Encoding with BERT: 100%|██████████| 58287/58287 [28:14<00:00, 34.40it/s]


item_entity_names embedding shape: (3730329, 768)


In [ ]:
!git add .